# 室内视频 → 3DGS 泼溅网页：分步 Notebook

本 Notebook 是 [pipeline.py](pipeline.py) 的可交互版本：每一步一个 cell，参数与命令完全一致，可单独重跑。

| 步骤 | 作用 | 产物 |
|---|---|---|
| `extract` | 从 mp4 均匀抽 N 张全景帧 | `recon/<name>/images/e_*.jpg` |
| `sfm` | Spirula sfm auto（全景自动转透视） | `recon/<name>/ws/sparse/0` |
| `dataset` | 目录联接（junction）组织训练集 | `tools/spirula/ds_<name>` |
| `train` | 训练 3DGS | `tools/spirula/out/run_<name>/step-*.ckpt` |
| `export` | ckpt → 单文件网页 | `splat_view.html` / `splat_view_mobile.html` |

**执行顺序**：从上往下依次运行。几个注意事项：

- `extract` 会先清掉 `images/` 里的旧 jpg，换帧数不用手动清理。
- `sfm` 若已有结果，会校验「帧集合指纹」；帧或视频变了必须先删 `ws/` 再重跑。
- `train` 要求输出目录为空；重跑前删目录，或改 `CFG["runname"]`。
- 训练耗时较长，`run()` 打印到「查看器仍在运行」时会主动结束子进程。


## 0. 参数与路径

所有可调参数集中在这里，默认值与 `pipeline.py` 当前默认值一致（160 帧 / 12000 迭代 / cap 240 万）。


In [ ]:
import json
import os
import subprocess
import sys
import time

# ---------------------------------------------------------------- 根路径
ROOT = r"d:\建模"
EXE = os.path.join(ROOT, "tools", "spirula", "app", "spirula.exe")
SPIRULA_OUT = os.path.join(ROOT, "tools", "spirula")

# ---------------------------------------------------------------- 可调参数
CFG = dict(
    video=os.path.join(ROOT, "VID_20261024_115439_00_959.mp4"),  # 原片 mp4
    name="joint",          # 任务名，决定目录与输出名
    frames=160,            # 抽帧数量
    width=2880,            # 抽帧分辨率（全景 2:1）
    height=1440,
    iters=12000,           # 3DGS 训练迭代数
    cap=2400000,           # 训练高斯上限
    sh=3,                  # 训练 SH 阶数
    resdiv=2,              # 训练分辨率降采样倍数
    quality="high",        # low / medium / high / ultra
    dtype="video",         # individual / video / internet：连续视频用 video
    overlap=15,            # 顺序匹配的相邻帧数
    camera_model="equirectangular",  # 360 全景；鱼眼用 OPENCV_FISHEYE
    floater="strong",      # off / mild / strong：漂浮团块清理强度
    runname="",            # 输出目录名，空则 run_<name>
    export_sh=1,           # 导出网页的 SH 阶数（1 阶约 100 MB，3 阶约 260 MB）
    mobile_splats=1200000, # 手机版保留的高斯数（按不透明度挑选）
)

# ---------------------------------------------------------------- 派生路径
NAME = CFG["name"]
REC = os.path.join(ROOT, "recon", NAME)
IMG = os.path.join(REC, "images")
WS = os.path.join(REC, "ws")
DS = os.path.join(SPIRULA_OUT, "ds_" + NAME)
RUN = CFG["runname"] or ("run_" + NAME)
OUT = os.path.join(SPIRULA_OUT, "out", RUN)

assert os.path.exists(EXE), "找不到 Spirula: " + EXE
os.makedirs(REC, exist_ok=True)

print("REC =", REC)
print("DS  =", DS)
print("OUT =", OUT)


## 1. 公共工具函数

移植自 `pipeline.py`：日志双写、流式执行外部命令、Windows junction。


In [ ]:
LOG = None  # 日志句柄，由下面的 open 赋值


def logline(s=""):
    """同时写终端与 recon/<name>/pipeline.log，逐行立即刷出"""
    print(s, flush=True)
    if LOG is not None:
        LOG.write(str(s) + "\n")
        LOG.flush()


def run(cmd, allow_fail=False, stop_on=None):
    """流式执行外部命令：逐行转发 stdout/stderr，记录耗时与退出码。

    stop_on: 看到该关键字表示任务已完成（Spirula 训练结束后会停在查看器循环里
    等待 Ctrl-C），此时主动结束子进程，避免 Notebook 永久挂起。
    """
    logline(">> " + " ".join(cmd))
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace", bufsize=1)
    stopped = False
    keys = stop_on if isinstance(stop_on, (list, tuple)) else ((stop_on,) if stop_on else ())
    for line in p.stdout:
        logline(line.rstrip())
        hit = next((k for k in keys if k in line), None)
        if hit:
            stopped = True
            logline("检测到 '%s'：任务已完成，结束查看器进程" % hit)
            p.terminate()
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                p.kill()
                p.wait()
            break
    rc = p.wait() if not stopped else 0
    logline("<< exit=%d%s  %.1fs" % (rc, " (任务完成后主动结束)" if stopped else "", time.time() - t0))
    if rc != 0 and not allow_fail:
        sys.exit(rc)


def junction(link, target):
    """Windows 目录联接：训练集不复制数据，只引用 images/ 与 sparse/0"""
    # 先校验目标，再动旧链接——否则目标缺失时会把原本完好的链接删掉
    if not os.path.exists(target):
        sys.exit("链接目标不存在：" + target)
    if os.path.lexists(link):
        is_j = os.path.islink(link) or getattr(os.path, "isjunction", lambda p: False)(link)
        if not is_j:
            sys.exit("已存在且不是链接，拒绝覆盖：" + link)
        r = subprocess.run(["cmd", "/c", "rmdir", link], capture_output=True, text=True)
        if r.returncode != 0:
            sys.exit("删除旧链接失败：%s\n%s" % (link, r.stderr))
    os.makedirs(os.path.dirname(link), exist_ok=True)
    r = subprocess.run(["cmd", "/c", "mklink", "/J", link, target], capture_output=True, text=True)
    if r.returncode != 0:
        sys.exit("创建链接失败：%s -> %s\n%s" % (link, target, r.stderr))
    logline("link %s -> %s" % (link, target))


def ckpt_step(name):
    """从 step-000012000.ckpt 里取出迭代数，用于挑最新 checkpoint"""
    try:
        return int(name.split("-")[1].split(".")[0])
    except Exception:
        return -1


# 打开日志并写开头（追加模式，多次运行都保留）
LOG = open(os.path.join(REC, "pipeline.log"), "a", encoding="utf-8")
logline("\n==== notebook  name=%s  %s ====" % (NAME, time.strftime("%Y-%m-%d %H:%M:%S")))
logline("video=%s frames=%d iters=%d cap=%d sh=%d resdiv=%d"
        % (CFG["video"], CFG["frames"], CFG["iters"], CFG["cap"], CFG["sh"], CFG["resdiv"]))


## 2. extract：从原片均匀抽帧

要点：

- **顺序解码**：`seek` 每次都要从头解码，长视频随机访问比顺序慢 10 倍以上。
- 抽帧位置均匀分布在 `[0, n-1]`，避免片头片尾偏移。
- 用 `buf.tofile()` 写盘，`cv2.imwrite` 不支持中文路径。


In [ ]:
def step_extract():
    """mp4 -> recon/<name>/images/e_0000.jpg ... （均匀抽 CFG["frames"] 张）"""
    import cv2
    if not os.path.exists(CFG["video"]):
        sys.exit("视频不存在：" + CFG["video"])

    os.makedirs(IMG, exist_ok=True)
    # 换帧数时清掉旧帧，避免两套时间点混在一起
    for f in os.listdir(IMG):
        if f.endswith(".jpg"):
            os.remove(os.path.join(IMG, f))

    cap = cv2.VideoCapture(CFG["video"])
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if n <= 0:
        cap.release()
        sys.exit("无法读取视频帧数：" + CFG["video"])
    logline("video frames=%d fps=%.2f dur=%.1fs" % (n, fps, n / (fps or 1)))

    want = min(CFG["frames"], n)
    if want != CFG["frames"]:
        logline("请求 %d 帧，视频只有 %d 帧，按 %d 帧抽取" % (CFG["frames"], n, want))

    # 均匀取 n-1 上的等间距帧号；取整后可能重复，用 set 去重
    want_set = set(int(round(i * (n - 1) / max(1, want - 1))) for i in range(want))
    if len(want_set) != want:
        logline("警告：抽帧去重后只有 %d 帧（请求 %d）" % (len(want_set), want))

    saved = 0
    # 顺序解码：一次性从头走到尾，只对目标帧做 retrieve
    for fi in range(n):
        ok = cap.grab()
        if not ok:
            break
        if fi not in want_set:
            continue
        ok, im = cap.retrieve()
        if not ok:
            continue
        # 统一缩放到 2880×1440（全景 2:1），INTER_AREA 缩小的画质最好
        if (im.shape[1], im.shape[0]) != (CFG["width"], CFG["height"]):
            im = cv2.resize(im, (CFG["width"], CFG["height"]), interpolation=cv2.INTER_AREA)
        p = os.path.join(IMG, "e_%04d.jpg" % saved)
        ok, buf = cv2.imencode(".jpg", im, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
        if not ok:
            sys.exit("encode failed at frame %d" % fi)
        buf.tofile(p)  # cv2.imwrite 不支持中文路径
        saved += 1
        if saved % 50 == 0:
            logline("  %d/%d" % (saved, want))

    cap.release()
    if saved == 0:
        sys.exit("一帧都没抽出来：" + CFG["video"])
    logline("extract -> %s  %d frames" % (IMG, saved))


step_extract()


## 3. sfm：稀疏重建（位姿 + 稀疏点云）

`spirula sfm auto` 内部完成：全景拆分透视 → 特征提取 → 顺序匹配（带回环闭合）→ 增量式 SfM。

- `--data-type video`：连续视频走顺序匹配，比 exhaustive 快很多。
- `--overlap 15`：相邻 15 帧参与匹配。
- 已有结果时会比对帧集合指纹，不一致直接报错，避免用旧位姿训练新帧。


In [ ]:
SFM_QUALITY = {"ultra": "extreme"}  # sfm 最高档叫 extreme，训练叫 ultra


def frame_sig():
    """当前帧集合的指纹：帧数 + 总字节数 + 视频文件信息"""
    fs = sorted(f for f in os.listdir(IMG) if f.endswith(".jpg"))
    st = os.stat(CFG["video"])
    return {"n": len(fs),
            "bytes": sum(os.path.getsize(os.path.join(IMG, f)) for f in fs),
            "video": os.path.abspath(CFG["video"]), "vbytes": st.st_size, "vmtime": int(st.st_mtime)}


def step_sfm():
    """images/ -> recon/<name>/ws/sparse/0（位姿）"""
    if not os.path.isdir(IMG) or not any(f.endswith(".jpg") for f in os.listdir(IMG)):
        sys.exit("抽帧目录为空：" + IMG)

    sig = frame_sig()
    sigp = os.path.join(REC, "sfm_sig.json")
    has_sfm = os.path.exists(os.path.join(WS, "sparse", "0", "images.bin"))
    if has_sfm:
        old = None
        if os.path.exists(sigp):
            try:
                old = json.load(open(sigp, encoding="utf-8"))
            except Exception:
                old = None
        if old is not None and old != sig:
            sys.exit("已有 SfM 结果与当前帧集合/视频不一致：\n  旧 %s\n  新 %s\n请先删除 %s 再重跑"
                     % (old, sig, WS))
        logline("发现已有 SfM 结果，沿用现有 sparse/0（若想重新解算，先删 %s）" % WS)

    os.makedirs(WS, exist_ok=True)
    cmd = [EXE, "sfm", "auto", IMG, "-o", WS,
           "--quality", SFM_QUALITY.get(CFG["quality"], CFG["quality"]),
           "--data-type", CFG["dtype"], "--overlap", str(CFG["overlap"]),
           "--no-masks"]
    if CFG["camera_model"]:
        cmd += ["--camera-model", CFG["camera_model"]]
    run(cmd)

    if not os.path.exists(os.path.join(WS, "sparse", "0", "images.bin")):
        sys.exit("SfM 没有产出 sparse/0/images.bin：" + WS)
    # 解算成功才更新指纹
    with open(sigp, "w", encoding="utf-8") as f:
        json.dump(sig, f, ensure_ascii=False)


step_sfm()


## 4. dataset：组织训练集目录

用 junction 把 `images/` 和 `sparse/0` 挂到 `tools/spirula/ds_<name>`，不复制数据。


In [ ]:
def step_dataset():
    """建立 tools/spirula/ds_<name> 目录结构（junction 指向真实数据）"""
    junction(os.path.join(DS, "images"), IMG)
    junction(os.path.join(DS, "sparse", "0"), os.path.join(WS, "sparse", "0"))
    logline("dataset -> %s" % DS)


step_dataset()


## 5. train：训练 3DGS

- `--train-resolution-divisor 2`：训练时图像降采样 2 倍，8GB 显存的关键。
- `--floater-suppression strong`：压制漂浮团块。
- 训练结束后 Spirula 会停在查看器循环里，`run()` 看到「查看器仍在运行」就主动结束进程。
- `run_joint` 已有训练结果，此 cell 会拒绝覆盖：只想重新导出网页就直接跑下一节 `export`；要重新训练请把 `CFG["runname"]` 改成新名字（如 `run_joint2`）。


In [ ]:
def step_train():
    """数据集 -> tools/spirula/out/run_<name>/step-*.ckpt/splat.ply"""
    if not os.path.isdir(os.path.join(DS, "images")):
        sys.exit("数据集未就绪，请先跑 dataset 步：" + DS)
    if not os.path.exists(os.path.join(DS, "sparse", "0", "images.bin")):
        sys.exit("数据集缺 sparse/0：" + DS)
    # 训练不会覆盖已有结果，必须先删或换 runname
    if os.path.isdir(OUT) and os.listdir(OUT):
        sys.exit("输出目录已存在训练结果：%s\n换 runname，或确认不要后手动删除该目录" % OUT)

    os.makedirs(os.path.join(SPIRULA_OUT, "out"), exist_ok=True)
    run([EXE, "train", "3dgs", "--data", DS,
         "--output-dir-prefix", os.path.join(SPIRULA_OUT, "out"),
         "--output-dir-name", RUN,
         "--quality", CFG["quality"],
         "--colmap-recon-dir", "sparse/0",
         "--floater-suppression", CFG["floater"],
         "--num-iterations", str(CFG["iters"]), "--cap-max", str(CFG["cap"]),
         "--sh-degree", str(CFG["sh"]),
         "--train-resolution-divisor", str(CFG["resdiv"])],
        stop_on="查看器仍在运行")


step_train()


## 6. export：ckpt → 单文件网页

`splat_ply_to_web.py` 把 `splat.ply` 压成二进制载荷内嵌进 HTML（Spirula 官方 WebGL2 渲染器），双击即可打开。

输出两份：

| 文件 | 高斯数 | 分辨率上限 | 用途 |
|---|---|---|---|
| `splat_view.html` | 全量 | 2160 / DPR 2 | 电脑 |
| `splat_view_mobile.html` | `mobile_splats` | 1440 / DPR 1.5 | 手机（大摇杆） |

找不到 `step-<iters>.ckpt` 时，会自动挑迭代数最大的 checkpoint。


In [ ]:
def step_export():
    """ckpt -> out/run_<name>/splat_view.html + splat_view_mobile.html"""
    src = os.path.join(WS, "sparse", "0")
    if not os.path.exists(os.path.join(src, "images.bin")):
        sys.exit("缺少 SfM 结果：" + src)
    if not os.path.isdir(OUT):
        sys.exit("训练输出目录不存在：" + OUT)

    # 优先用指定迭代数的 checkpoint，没有就取迭代数最大的
    ck = os.path.join(OUT, "step-%09d.ckpt" % CFG["iters"])
    if not os.path.isdir(ck):
        cands = [f for f in os.listdir(OUT)
                 if f.endswith(".ckpt") and os.path.isdir(os.path.join(OUT, f))]
        if not cands:
            sys.exit("没有找到 checkpoint：" + OUT)
        ck = os.path.join(OUT, max(cands, key=ckpt_step))
        logline("使用 checkpoint：%s" % ck)

    ply = os.path.join(ck, "splat.ply")
    if not os.path.exists(ply):
        sys.exit("缺 splat.ply: " + ply)

    gen = os.path.join(ROOT, "scripts", "splat_ply_to_web.py")
    tpl = os.path.join(ROOT, "scripts", "splat_tpl2.html")
    base = [sys.executable, gen, "--ply", ply, "--sparse", src,
            "--tmpl", tpl, "--sh_degree", str(CFG["export_sh"])]

    # 电脑版：全量高斯，2160 宽上限
    run(base + ["--out", os.path.join(OUT, "splat_view.html"),
                "--maxsplats", "0", "--res_cap", "2160", "--dpr_cap", "2"])

    # 手机版：裁剪高斯 + 1440 宽上限 + 大摇杆
    run(base + ["--out", os.path.join(OUT, "splat_view_mobile.html"),
                "--maxsplats", str(CFG["mobile_splats"]), "--res_cap", "1440",
                "--dpr_cap", "1.5", "--mobile"])


step_export()


## 7. 结果检查

跑完后确认产物齐全。再次运行某个步骤时，直接跳到对应 cell 即可；`extract → sfm → dataset` 有变更时需要按顺序重跑。


In [ ]:
checks = [
    ("抽帧", IMG),
    ("SfM", os.path.join(WS, "sparse", "0", "images.bin")),
    ("训练输出", OUT),
    ("电脑版网页", os.path.join(OUT, "splat_view.html")),
    ("手机版网页", os.path.join(OUT, "splat_view_mobile.html")),
]
for label, p in checks:
    ok = os.path.exists(p)
    size = ""
    if ok and os.path.isfile(p):
        size = "  %.1f MB" % (os.path.getsize(p) / 1048576)
    print(("OK   " if ok else "缺失 ") + label + "： " + p + size)
